# T01. One line, seven stages

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/t01-one-line-seven-stages/t01.ipynb)

This lesson follows one line of Python, `answer = 6 * 7`, from the bytes in a file to the number 42 sitting in a namespace. There are seven stages in between, and you can watch every one of them from inside Python, with no C compiler and no CPython build of your own.

You do not need to know any C to do this, and you do not need to already know what a compiler is. By the end you will have watched CPython throw your multiplication away before your program ever ran, and you will know which file and which function threw it away.

Everything below is executed in continuous integration on CPython 3.15.0rc1 and on 3.14 before it reaches you. Where the two disagree the lesson says so and prints what your build actually did, rather than asking you to trust a number that was true on somebody else's machine.

## How to read the source references

References to CPython's own source look like `Python/ceval.c:1213@v3.15.0rc1#_PyEval_EvalFrameDefault`. That is a file, a line or a line range, the exact release tag the line numbers belong to, and the name of the function containing them. Every reference in this repository is a link you can click, and every one is resolved against the pinned source tree on every change, so a reference that has drifted fails the build instead of quietly pointing you at the wrong function.

The trailing symbol name is the part that makes this work. A bare line number goes stale silently when somebody upstream inserts a function above it, and it then points at something plausible and wrong, which is worse than pointing at nothing.

## Setup

Colab does not come with the small instrumentation package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing at all.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which interpreter is this

Nearly every fact in this lesson is a fact about one particular build of CPython. Instruction names change between releases, so do the sizes of things, so the first thing every lesson here does is say out loud which binary is about to produce the output you are reading.

In [ ]:
import pyxray

pyxray.show()

## The line

Here it is. One assignment, two numbers, one multiplication.

In [ ]:
SOURCE = "answer = 6 * 7\n"

print(SOURCE)

## Stage 1. Bytes become tokens

CPython reads your file as bytes and cuts them into tokens: a name, an equals sign, a number, an operator, another number, an end of line. The code that does the cutting is [Parser/lexer/lexer.c:1626-1635@v3.15.0rc1#_PyTokenizer_Get](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/lexer/lexer.c#L1626-L1635), and it is a hand written lexer rather than anything generated by a tool.

The standard library exposes the same tokenizer through the `tokenize` module, so the stream below is the real one and not an imitation of it.

In [ ]:
import token

from pyxray import compiler

for item in compiler.tokens(SOURCE):
    name = token.tok_name[item.type]
    line, start = item.start[0], item.start[1]
    print(f"{name:<10} {item.string!r:<10} line {line}, columns {start} to {item.end[1]}")

Two of those tokens are not in your file. `ENCODING` comes first and carries the tokenizer's answer to the question it has to settle before it can read a single character of Python, which is how these bytes are to be decoded at all. `ENDMARKER` comes last and marks the end of input. Neither one was written by you.

Indentation is invented in the same way, and it is easier to see with a line that has some.

In [ ]:
for item in compiler.tokens("if answer:\n    print(answer)\n"):
    print(f"{token.tok_name[item.type]:<10} {item.string!r}")

`INDENT` and `DEDENT` appear nowhere in that text. The tokenizer synthesizes them from column positions and hands the parser something that behaves exactly like the braces other languages make you type. That is the whole of Python's significant whitespace, decided here, in the first stage, and never thought about again.

## Stage 2. Tokens become a tree

The parser reads the token stream and builds an abstract syntax tree. CPython has used a PEG parser since 3.9, generated from a grammar file into `Parser/parser.c`, and the function that drives it is [Parser/pegen.c:938-941@v3.15.0rc1#_PyPegen_run_parser](https://github.com/python/cpython/blob/v3.15.0rc1/Parser/pegen.c#L938-L941).

The tree for a single assignment is small enough to read whole.

In [ ]:
import ast

tree = ast.parse(SOURCE)

print(ast.dump(tree, indent=4))

Read it from the inside out. Two `Constant` nodes hold 6 and 7. A `BinOp` holds those two with a `Mult` between them. An `Assign` puts the result into a `Name` whose context is `Store`, meaning this name is being written rather than read. Nothing has been evaluated yet. `6 * 7` is still a multiplication of two constants sitting in a tree.

The tree is an ordinary Python object, which means you can change it and compile the result. This is the first moment where the pipeline stops being a black box.

In [ ]:
edited = ast.parse(SOURCE)
edited.body[0].value.right = ast.Constant(value=8)

namespace = {}
exec(compile(ast.fix_missing_locations(edited), "<edited>", "exec"), namespace)

print(namespace["answer"])

## Stage 3. The tree gets a symbol table

Before generating a single instruction, CPython walks the tree and decides what every name in it is. Is `answer` a local, a global, or something borrowed from an enclosing function? The compiler cannot choose an instruction for a name until it knows, because a local is a numbered slot in a frame and a global is a dictionary lookup by name, and those are different opcodes with very different costs. The pass that decides is [Python/symtable.c:415-418@v3.15.0rc1#_PySymtable_Build](https://github.com/python/cpython/blob/v3.15.0rc1/Python/symtable.c#L415-L418).

In [ ]:
print(compiler.symbols(SOURCE).tree())

`answer` comes back as both local and global, which looks like a contradiction and is not one. At module level a name really is stored in this module's own namespace, so it is local to this scope, and it really is the thing every function in the file will find when it reads that name, so it is a global. Both answers are true at once. Most explanations of Python scope pick one of them and quietly drop the other.

Inside a function the two come apart, which is easier to see than to describe.

In [ ]:
print(compiler.symbols("def f(a):\n    return a + answer\n").tree())

Now `a` is a parameter and a local and nothing else, and `answer` is a global that is only ever read. There is also a scope called `__annotate__` that you did not write. That is PEP 649, which made annotations lazy in 3.14: every `def` gets a hidden function that would compute its annotations if anything ever asked for them, and it gets one whether or not there are any annotations to compute.

## Stage 4. The tree becomes instructions

Now the compiler walks the tree a second time and emits instructions, a small pile of them per node. The case that handles an expression is [Python/codegen.c:894-897@v3.15.0rc1#_PyCodegen_Expression](https://github.com/python/cpython/blob/v3.15.0rc1/Python/codegen.c#L894-L897).

This is usually where a stock interpreter stops being observable from Python, and it is where this project's favourite trick comes in. CPython ships a module called `_testinternalcapi` that exposes the compiler's stages one at a time, so you can hold the instruction sequence in your hands before the optimizer has touched it. If the next cell raises, read the message: it means your interpreter was built without those hooks, and it will tell you what still works.

In [ ]:
result = compiler.stages(SOURCE)

for item in result.codegen:
    print(item)

`LOAD_CONST 0`, `LOAD_CONST 1`, `BINARY_OP 5`. The multiplication is still there and, as far as this stage is concerned, it is still going to happen while your program runs. `BINARY_OP 5` is multiply, 5 being the position of `*` in CPython's table of binary operators.

`ANNOTATIONS_PLACEHOLDER` is marked pseudo. Pseudo instructions exist only inside the compiler, as markers for later passes, and none of them ever reaches a code object. This one holds the spot where the module's annotations would be set up if it had any.

## Stage 5. The optimizer rewrites the instructions

The instruction sequence is turned into a control flow graph, and the graph is optimized. The entry point is [Python/flowgraph.c:3753-3757@v3.15.0rc1#_PyCfg_OptimizeCodeUnit](https://github.com/python/cpython/blob/v3.15.0rc1/Python/flowgraph.c#L3753-L3757). Here is what it did, with the two sequences side by side.

In [ ]:
print(compiler.what_the_optimizer_did(result))

Look at the right hand column. `LOAD_CONST`, `LOAD_CONST`, `BINARY_OP` has become a single `LOAD_SMALL_INT 42`. The multiplication is gone. It was performed once, at compile time, by [Python/flowgraph.c:1860-1866@v3.15.0rc1#eval_const_binop](https://github.com/python/cpython/blob/v3.15.0rc1/Python/flowgraph.c#L1860-L1866), and the answer was written straight into the instruction stream. Your program will never multiply anything.

That is constant folding, and CPython does it twice, in two places, on two different data structures. The one you just watched works on the control flow graph. There is an earlier one that works on the tree, [Python/ast_preprocess.c:370-383@v3.15.0rc1#fold_binop](https://github.com/python/cpython/blob/v3.15.0rc1/Python/ast_preprocess.c#L370-L383), and it is invisible in this notebook for a reason worth knowing: the stage hook is handed a tree straight from `ast.parse`, which skips the preprocessing pass that a real `compile()` call would have run first. So what you are looking at is the second folder catching what the first one was never asked to look at.

Folding only happens when the compiler can see both operands for itself. Replace one of the numbers with a name and the multiplication survives all the way into the finished code, because `six` could be anything at all by the time that line runs.

In [ ]:
print(compiler.what_the_optimizer_did(compiler.stages("answer = six * 7\n")))

## Stage 6. The instructions become a code object

The assembler turns the optimized graph into the immutable object CPython actually executes, built by [Objects/codeobject.c:715-718@v3.15.0rc1#_PyCode_New](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/codeobject.c#L715-L718). The whole front end, every stage above, is driven from [Python/compile.c:1526-1540@v3.15.0rc1#_PyAST_Compile](https://github.com/python/cpython/blob/v3.15.0rc1/Python/compile.c#L1526-L1540).

In [ ]:
import dis

print("co_consts    ", result.code.co_consts)
print("co_names     ", result.code.co_names)
print("co_stacksize ", result.code.co_stacksize)
print("co_code      ", len(result.code.co_code), "bytes")
print()
dis.dis(result.code)

Two details in there repay a second look, and the next cell makes both of them concrete.

The constant table still contains 6. Nothing loads it. The table was built while the multiplication still existed, and it was not pruned afterwards, so a number your program has no use for is carried around inside the code object for as long as the code object lives.

The 42 is not in the constant table at all. It is the argument of `LOAD_SMALL_INT`, sitting inside the instruction itself. Small integers do not need an entry because CPython keeps them alive permanently anyway, which is a lesson of its own.

In [ ]:
loaded = {i.argval for i in dis.get_instructions(result.code) if i.opname == "LOAD_CONST"}
inline = [str(i) for i in result.optimized if "SMALL_INT" in i.opname]

print("constants the code object carries:", result.code.co_consts)
print("constants any instruction loads:  ", loaded or "none")
print("where the 42 actually lives:      ", inline)

The last two instructions are the module's implicit `return None`, and they are spelled differently depending on which build you are on. From 3.15 there is a `LOAD_COMMON_CONSTANT`, which pulls `None` out of a fixed table shared by every code object, so `None` no longer needs an entry of its own and the constant table is one shorter. On 3.14 it is still an ordinary `LOAD_CONST`. The bytecode is a couple of bytes longer on 3.15 for an unrelated reason: `RESUME` grew an inline cache entry.

Rather than take either version's word for it, print what your build did.

In [ ]:
print("python            ", sys.version.split()[0])
print("last instructions ", [str(item) for item in result.optimized[-2:]])
print("co_consts         ", result.code.co_consts)
print("bytecode          ", len(result.code.co_code), "bytes")
print("None is a constant", None in result.code.co_consts)

## Stage 7. The code object runs

Finally the evaluation loop executes the code object one instruction at a time, in [Python/ceval.c:1213@v3.15.0rc1#_PyEval_EvalFrameDefault](https://github.com/python/cpython/blob/v3.15.0rc1/Python/ceval.c#L1213), which is the largest switch statement most people will ever meet.

In [ ]:
namespace = {}
exec(result.code, namespace)

print(namespace["answer"])

42, and your program never multiplied anything.

Here is the whole trip in one line of counts.

In [ ]:
print(result.summary())

## What just happened

Source text was decoded and cut into tokens. Tokens were parsed into a tree. The tree was walked once to decide what every name meant, and walked again to emit instructions. The instructions became a graph, the graph was optimized, and the optimizer did your arithmetic for you. What was left was assembled into a code object, and the code object was executed.

Seven stages, one line of Python, and the only one most people ever think about is the last.

## Try it yourself

Change `MINE` below and run the cell. Some things worth trying, roughly in order of how surprising the answer is.

Try `x = 2 ** 10`, which folds, and then `x = 2 ** 10000`, which does not. The optimizer refuses to fold a result that would be enormous, because the constant would have to be stored in every copy of the code object forever.

Try `x = "ab" * 3` and then `x = "ab" * 100000`, and watch the same size limit apply to strings.

Try `x = 1 / 0`. The compiler will not fold something that raises, so the division survives to runtime, which is why you get a traceback pointing at your line rather than a compile error.

Try `if True:\n    x = 1`, and see how much of the `if` is left by the time the optimizer has finished.

In [ ]:
MINE = "x = 2 ** 10\n"

mine = compiler.stages(MINE)
print(mine.summary())
print()
print(compiler.what_the_optimizer_did(mine))

## Where this goes next

The seven stages are the map for the rest of the material. Each of them gets a lesson of its own, and each of those lessons is the same shape as this one: a small piece of Python you can run, the exact CPython source that does the work, and an experiment that fails if the explanation is wrong.

The next lesson takes the last stage apart. `LOAD_SMALL_INT 42` puts something on a stack, `STORE_NAME` takes it off again, and both of those sentences hide the reference counting that decides when the object underneath is allowed to disappear.